# Model Retention Walk-Forward Gate

Purpose: audit replayable SOXL models against the current retention target: complete walk-forward evidence, strict max drawdown below 20%, and YTD return aligned to both the converted SOXL backtest reference and current best replay model.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'AGENTS.md').exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / 'AGENTS.md').exists())
CARD = ROOT / 'runtime/model_selection/card/model_retention_walkforward_gate_v1'
CARD.mkdir(parents=True, exist_ok=True)
ROOT, CARD

(PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix'),
 PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/model_retention_walkforward_gate_v1'))

In [2]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.model_retention_gate import scan_model_retention_gates, write_retention_gate_outputs

result = scan_model_retention_gates(root=ROOT, max_walkforward_drawdown_pct=20.0, best_return_ratio=0.95)
write_retention_gate_outputs(result, output_dir=CARD)
summary = result['summary']
summary

{'generated_at': '2026-06-13T17:28:16.324652+00:00',
 'profiles_scanned': 12,
 'passing_profiles': [],
 'passing_profile_count': 0,
 'reference_profile': 'soxl_backtest_policy_reimplementation_v1',
 'reference_return_pct': 132.077978,
 'reference_max_drawdown_pct': 13.554089,
 'current_best_return_pct': 224.361539,
 'best_return_ratio': 0.95,
 'required_best_return_pct': 213.143462,
 'max_walkforward_drawdown_pct': 20.0,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']}

In [3]:
rows = pd.DataFrame(result['rows'])
display(rows[['profile', 'status', 'ytd_return_pct', 'ytd_max_drawdown_pct', 'wf_windows_present', 'worst_wf_max_drawdown_pct', 'gate_pass', 'retention_decision', 'failures']])

,profile,status,ytd_return_pct,ytd_max_drawdown_pct,wf_windows_present,worst_wf_max_drawdown_pct,gate_pass,retention_decision,failures
0,soxl_backtest_policy_reimplementation_v1,reference,132.077978,13.554089,,NaN,True,keep_reference_baseline,[]
1,balanced,archived,159.205721,29.698281,,NaN,False,retain_as_evidence_only,"[missing_wf_window:wf_2020, missing_wf_window:..."
2,balanced_macro_v1,candidate,196.749008,29.901822,,NaN,False,retain_as_evidence_only,"[missing_wf_window:wf_2020, missing_wf_window:..."
3,balanced_rule_residual_v19,archived,173.299227,24.399161,,NaN,False,retain_as_evidence_only,"[missing_wf_window:wf_2020, missing_wf_window:..."
4,smoke,canonical,224.361539,30.346650,,NaN,False,retain_as_evidence_only,"[missing_wf_window:wf_2020, missing_wf_window:..."
5,balanced_rule_residual_v22,canonical,116.882644,26.278174,"wf_2020,wf_2022,wf_2023_2024",54.665610,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, wf_max_drawdow..."
6,balanced_rule_residual_imitation_v4,candidate,145.031235,25.999446,,NaN,False,retain_as_evidence_only,"[missing_wf_window:wf_2020, missing_wf_window:..."
7,drawdown_rebound_reward_train_v1,ablation,172.832548,27.310377,"wf_2020,wf_2022,wf_2023_2024",23.546830,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, ytd_return_bel..."
8,mode_conditioned_rebound_reward_train_v2,ablation,177.573569,27.278641,"wf_2020,wf_2022,wf_2023_2024",21.831720,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, wf_max_drawdow..."
9,state_conditioned_drawdown_rebound_reward_trai...,ablation,196.505054,31.390931,"wf_2020,wf_2022,wf_2023_2024",27.608986,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, ytd_return_bel..."


## Conclusion

- `soxl_backtest_policy_reimplementation_v1` is retained as the converted rule-policy reference baseline.
- No replayable RL/model profile currently satisfies all gates: complete `wf_2020`, `wf_2022`, `wf_2023_2024`; every WF MaxDD `<20%`; YTD return `>=132.077978%`; and YTD return `>=95%` of the current best replay (`smoke`, `224.361539%`).
- This does not delete protected runtime actors. It turns the user target into an auditable retention gate for the next training/model-selection loop.
- Next action: train or select candidates against this gate; do not treat dashboard-visible replay models as final keepers until this card reports `passing_profile_count > 0`.